# Figuring out the best recipe
To figure out the recipe which produces the most amount of information with each
recycle (does not necessarily need to be the scrap recycle) take a look at the
data dump of all recipes.

In [17]:
import requests
import tabulate
import math
import json
import os
import numpy as np


In [ ]:
# Download the factorio recipe data and cache it locally.
# This piggybacks on kirkmcdonald's pre-processed data to avoid having to parse the raw factorio data ourselves.
url = "https://raw.githubusercontent.com/KirkMcDonald/kirkmcdonald.github.io/master/data/space-age-2.0.55.json"
file_name = "space-age-2.0.55.json"

if not os.path.exists(file_name):
    response = requests.get(url)
    with open(file_name, 'wb') as f:
        f.write(response.content)
    print(f"Downloaded and cached {file_name}")
else:
    print(f"{file_name} already exists in the cache")

In [ ]:
# Load the data from the cached file, show a sample recipe
with open(file_name) as f:
    data = json.load(f)

recipes = data["recipes"]
recipes[20]

{'allow_productivity': False,
 'category': 'recycling',
 'energy_required': 0.03125,
 'icon_col': 10,
 'icon_row': 12,
 'ingredients': [{'amount': 1, 'name': 'arithmetic-combinator'}],
 'key': 'arithmetic-combinator-recycling',
 'localized_name': {'en': 'Arithmetic combinator recycling'},
 'results': [{'amount': 1.25, 'name': 'copper-cable'},
  {'amount': 1.25, 'name': 'electronic-circuit'}],
 'subgroup': 'other'}

In [14]:
# Iterate over all the recycling recipes and figure out the entropy generated by the output
def generates_entropy(recipe):
    if any("probability" in res.keys() for res in recipe["results"]):
        return True
    if any(int(res["amount"]) != res["amount"] for res in recipe["results"]):
        return True
    return False


def bits_observable(p):
    if p == 0 or p == 1:
        return 0
    return p * math.floor(-math.log2(p)) + (1 - p) * math.floor(-math.log2(1 - p))


def recipe_entropy(recipe):
    assert generates_entropy(recipe), f"Recipe {recipe['key']} does not generate entropy"

    # Get the probabilities of each output
    outputs = recipe["results"]
    probs = []
    for output in outputs:
        if "probability" in output:
            probs.append(output["probability"])
        else:
            probs.append(output["amount"] - math.floor(output["amount"]))
    probs = np.array(probs)
    probs = np.minimum(probs, 1 - probs)

    # Calculate the observable entropy of the output
    return sum((p if p != 0.5 else 1) * bits_observable(p) for p in probs if p > 0)


def recipe_table_dict(recipes):
    return [
        {
            "key": r["key"],
            "entropy": round(recipe_entropy(r), 3),
            "crafting_speed": r["energy_required"],
            "outputs": r["results"],
        }
        for r in recipes
    ]


def recipe_table(recipes):
    table = recipe_table_dict(recipes)
    for row in table:
        row["entropy"] = round(row["entropy"], 3)
        row["outputs"] = ", ".join(
            f"{o['amount']}x {o['name']}" + (f" ({o['probability']:.1%})" if "probability" in o else "")
            for o in row["outputs"]
        )
    non_dict_table = [[row["key"], row["entropy"], row["crafting_speed"], row["outputs"]] for row in table]
    return tabulate.tabulate(
        [["Recipe", "Entropy", "Crafting Speed", "Outputs"], *non_dict_table],
        tablefmt="html",
    )

In [15]:
# Check if there are any recipes which have both a probability and a non-integer amount in one of the outputs
def has_mixed_output(recipe):
    if any("probability" in res.keys() for res in recipe["results"]):
        if any(int(res["amount"]) != res["amount"] for res in recipe["results"]):
            return True
    return False

mixed_recipes = filter(has_mixed_output, recipes)
recipe_table(mixed_recipes)

Recipe,Entropy,Crafting Speed,Outputs


In [16]:
prob_recipes = filter(generates_entropy, recipes)
recipe_entropies = [(r, recipe_entropy(r)) for r in prob_recipes]
recipe_entropies.sort(key=lambda x: (x[1], -x[0]['energy_required']), reverse=True)
recipe_entropies = [r for r, entropy in recipe_entropies]
data = recipe_table_dict(recipe_entropies)
with open("scrap-recycling-recipes.json", "w") as f:
    json.dump(data, f)
recipe_table(recipe_entropies)

Recipe,Entropy,Crafting Speed,Outputs
oil-refinery-recycling,4.125,0.5,"3.75x steel-plate, 2.5x iron-gear-wheel, 2.5x stone-brick, 2.5x electronic-circuit, 2.5x pipe"
electromagnetic-plant-recycling,4.0,0.625,"37.5x holmium-plate, 12.5x steel-plate, 12.5x processing-unit, 12.5x refined-concrete"
fusion-reactor-equipment-recycling,3.25,1.875,"0.25x fission-reactor-equipment, 2.5x fusion-power-cell, 62.5x tungsten-plate, 25.0x carbon-fiber, 6.25x supercapacitor, 62.5x quantum-processor"
overgrowth-jellynut-soil-recycling,3.125,0.625,"0.5x artificial-jellynut-soil, 1.25x jellynut-seed, 2.5x biter-egg, 12.5x spoilage"
overgrowth-yumako-soil-recycling,3.125,0.625,"0.5x artificial-yumako-soil, 1.25x yumako-seed, 2.5x biter-egg, 12.5x spoilage"
tesla-turret-recycling,3.125,1.875,"0.25x teslagun, 2.5x supercapacitor, 2.5x processing-unit, 12.5x superconductor"
explosive-cannon-shell-recycling,3.0,0.5,"0.5x steel-plate, 0.5x plastic-bar, 0.5x explosives"
foundry-recycling,3.0,0.625,"12.5x tungsten-carbide, 12.5x steel-plate, 7.5x electronic-circuit, 5.0x refined-concrete"
teslagun-recycling,3.0,1.875,"2.5x holmium-plate, 2.5x superconductor, 7.5x plastic-bar"
stack-inserter-recycling,2.25,0.03125,"0.25x bulk-inserter, 0.25x processing-unit, 0.5x carbon-fiber, 2.5x jelly"
